In [12]:
import pandas as pd
import numpy as np

df_analise = pd.read_csv("../data/processed/vendas_tratadas.csv", low_memory=False)
df_analise['Data'] = pd.to_datetime(df_analise['Data'])
df_analise.info()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Unnamed: 0      100000 non-null  int64         
 1   ID_Pedido       100000 non-null  int64         
 2   Data            100000 non-null  datetime64[us]
 3   Loja            100000 non-null  str           
 4   Produto         100000 non-null  str           
 5   Preco_Unitario  100000 non-null  float64       
 6   Qtd             100000 non-null  int64         
 7   Cliente         100000 non-null  str           
 8   Gerente         100000 non-null  str           
 9   Meta_Mensal     100000 non-null  float64       
 10  Faturamento     100000 non-null  float64       
 11  Ano_Mes         100000 non-null  str           
dtypes: datetime64[us](1), float64(3), int64(3), str(5)
memory usage: 9.2 MB


In [13]:
# KPIs Executivos Gerais
faturamento_total = df_analise['Faturamento'].sum()
ticket_medio = df_analise['Faturamento'].mean()
total_pedidos = df_analise['ID_Pedido'].nunique()

print(f"Faturamento Total: R$ {faturamento_total:,.2f}")
print(f"Ticket Médio: R$ {ticket_medio:,.2f}")
print(f"Total de Pedidos: {total_pedidos}")

Faturamento Total: R$ 299,472,330.00
Ticket Médio: R$ 2,994.72
Total de Pedidos: 100000


## 1. Ranking de Faturamento por Loja
**Pergunta de Negócio:** Quais filiais geram a maior receita total para a empresa e qual canal apresenta o menor desempenho financeiro?

In [14]:
analise_lojas = df_analise[["Loja", "Faturamento"]].groupby("Loja").sum().reset_index()
analise_lojas = analise_lojas.sort_values(by = "Faturamento", ascending=False)
display(analise_lojas) 

,Loja,Faturamento
6,Salvador,42300610.0
5,Rio De Janeiro,42294720.0
4,Recife,42190390.0
7,São Paulo,42090690.0
0,Belo Horizonte,41714890.0
3,Porto Alegre,41678460.0
1,Curitiba,41121720.0
2,Online,6080850.0



💡 **Insight:** Salvador e Rio de Janeiro lideram a receita total, enquanto o canal Online representa a menor fatia do faturamento acumulado.

## 2. Desempenho de Produtos no Canal Online
**Pergunta de Negócio:** Quais são os produtos mais vendidos (em volume de unidades) exclusivamente no e-commerce?

In [15]:


analise_prod_online = df_analise[df_analise["Loja"] == "Online"]
analise_online = analise_prod_online[["Produto","Qtd"]].groupby("Produto").sum().reset_index()
analise_online = analise_online.sort_values(by= "Qtd", ascending= False)
display(analise_online)

,Produto,Qtd
4,Notebook HP,442
0,Cabo HDMI,403
7,iPhone 14,390
2,Mouse Gamer,379
3,Notebook Dell,369
6,Teclado Mecânico,343
1,"Monitor 27""",332
5,Smartphone Samsung,311


💡 **Insight** **Notebook HP** (442 unidades) é o carro-chefe do site, seguido por **Cabo HDMI** (403 unidades) e **iPhone 14** (390 unidades), direcionando a prioridade para campanhas de marketing digital nesses itens.

## 3. Distribuição do Volume de Vendas por Loja e Produto
**Pergunta de Negócio:** Como a demanda unitária de cada produto se distribui entre as diferentes filiais da rede?

In [16]:

analise_lojas_prod = df_analise.groupby(["Loja", "Produto"])["Qtd"].sum().reset_index()
analise_lojas_prod = analise_lojas_prod.sort_values(by=["Produto","Qtd"], ascending=[True, False])
display(analise_lojas_prod)

,Loja,Produto,Qtd
40,Rio De Janeiro,Cabo HDMI,2747
8,Curitiba,Cabo HDMI,2698
56,São Paulo,Cabo HDMI,2649
0,Belo Horizonte,Cabo HDMI,2636
24,Porto Alegre,Cabo HDMI,2571
...,...,...,...
15,Curitiba,iPhone 14,2652
39,Recife,iPhone 14,2652
31,Porto Alegre,iPhone 14,2540
7,Belo Horizonte,iPhone 14,2478


💡 **Insight:** **Cabo HDMI** lidera o volume físico com destaque no Rio de Janeiro (2.747 unidades) e Curitiba (2.698 unidades). A discrepância para o e-commerce (apenas 403 unidades) auxilia no planejamento e na alocação regionalizada de estoque.

## 4. Atingimento de Metas e Performance Gerencial
**Pergunta de Negócio:** Qual foi a taxa percentual de cumprimento da meta mensal por loja e gerente, e quais unidades superaram o objetivo estipulado?

In [17]:

desempenho_metas = df_analise.groupby(['Loja', 'Gerente', 'Ano_Mes', 'Meta_Mensal'])['Faturamento'].sum().reset_index()
desempenho_metas['Bateu_Meta'] = np.where(desempenho_metas['Faturamento'] >= desempenho_metas['Meta_Mensal'], 'Sim', 'Não')
desempenho_metas['%_Atingimento'] = (desempenho_metas['Faturamento'] / desempenho_metas['Meta_Mensal']) * 100
desempenho_metas['%_Atingimento'] = desempenho_metas['%_Atingimento'].round(2)
desempenho_metas['Status_Meta'] = np.where(desempenho_metas['%_Atingimento'] >= 100, 'Meta Batida', 'Abaixo da Meta')
desempenho_metas = desempenho_metas.sort_values(by=['Loja','Ano_Mes',], ascending=False)
display(desempenho_metas)

,Loja,Gerente,Ano_Mes,Meta_Mensal,Faturamento,Bateu_Meta,%_Atingimento,Status_Meta
191,São Paulo,Carlos,2024-12,50000.0,1744030.0,Sim,3488.06,Meta Batida
190,São Paulo,Carlos,2024-11,50000.0,1920100.0,Sim,3840.20,Meta Batida
189,São Paulo,Carlos,2024-10,50000.0,1889390.0,Sim,3778.78,Meta Batida
188,São Paulo,Carlos,2024-09,50000.0,1618540.0,Sim,3237.08,Meta Batida
187,São Paulo,Carlos,2024-08,50000.0,1764270.0,Sim,3528.54,Meta Batida
...,...,...,...,...,...,...,...,...
4,Belo Horizonte,Juliana,2023-05,55000.0,1997800.0,Sim,3632.36,Meta Batida
3,Belo Horizonte,Juliana,2023-04,55000.0,1838640.0,Sim,3342.98,Meta Batida
2,Belo Horizonte,Juliana,2023-03,55000.0,1871600.0,Sim,3402.91,Meta Batida
1,Belo Horizonte,Juliana,2023-02,55000.0,1832950.0,Sim,3332.64,Meta Batida


    💡 **Insight:** A meta mensal atual (R$ 50.000,00) encontra-se defasada em relação ao potencial real das filiais. Unidades como São Paulo superaram 3.000% da meta no período, indicando a necessidade de reajuste do planejamento orçamentário.